# KO CBF Generation
Generate a number of keep-out (KO) regions centered on and around the path with feasible velocities and accelerations which will be used to train the controller while still producing a guaranteed safe output at all times.

In [31]:
import pathlib
import h5py

import numpy as np

import itertools

In [32]:
episodes_train_dir = pathlib.Path("../data/episodes/train")
episodes_test_dir = pathlib.Path("../data/episodes/test")

In [33]:
# load all the episodes from file into memory


def load_from_h5(h5_file_dir):
    episodes = []
    files = []
    for file in h5_file_dir.iterdir():
        if file.name == ".gitkeep":
            continue  # don't try to load this one
        episode = dict()
        with h5py.File(file, "r") as f:
            f.visititems(lambda name, obj: episode.update({name: np.asarray(obj)}))
            episode.update(f.attrs)
        episodes.append(episode)
        files.append(file.name)
    return episodes, files


train_episodes, train_files = load_from_h5(episodes_train_dir)
test_episodes, test_files = load_from_h5(episodes_test_dir)

train_episodes[0]

{'ddx_traj': array([-0.16974615, -0.16726077, -0.16477539, -0.16229   , -0.15980462,
        -0.15731924, -0.15483385, -0.15234847, -0.14986308, -0.1473777 ,
        -0.14489232, -0.14240693, -0.13992155, -0.13743616, -0.13495078,
        -0.1324654 , -0.12998001, -0.12749463, -0.12500925, -0.12252386,
        -0.12003848, -0.11755309, -0.11506771, -0.11258233, -0.11009694,
        -0.10761156, -0.10512617, -0.10264079, -0.10015541, -0.09767002,
        -0.09518464, -0.09269926, -0.09021387, -0.08772849, -0.0852431 ,
        -0.08275772, -0.08027234, -0.07778695, -0.07530157, -0.07281618,
        -0.0703308 , -0.06784542, -0.06536003, -0.06287465, -0.06038927,
        -0.05790388, -0.0554185 , -0.05293311, -0.05044773, -0.04796235,
        -0.04547696, -0.04299158, -0.04050619, -0.03802081, -0.03553543,
        -0.03305004, -0.03056466, -0.02807928, -0.02559389, -0.02310851,
        -0.02062312, -0.01813774, -0.01565236, -0.01316697, -0.01068159,
        -0.00819621, -0.00571082, -0.00

In [34]:
num_ko_regions = 10  # number of KO regions to generate for each trajectory

# standard deviations (zero mean unless tuple)
dist_ko_offsets = 0.5 * np.diag([1.0, 1.0])
dist_vel_x = 0.5
dist_vel_y = dist_vel_x
dist_accel_x = 0.1
dist_accel_y = dist_accel_x

dist_radius = 1.1, 0.4  # mean, deviation
dist_vel_radius = 0.3
dist_accel_radius = 0.1

r = np.random.default_rng(42)  # for reproducibility

In [35]:
keep_out_train_dir = pathlib.Path("../data/keep_out/train")
keep_out_test_dir = pathlib.Path("../data/keep_out/test")


# empty these repositories so we have no issues when writing the episodes that
# were generated in the preceding section
for file in itertools.chain(keep_out_train_dir.iterdir(), keep_out_test_dir.iterdir()):
    if file.name == ".gitkeep":
        continue  # don't delete this file
    file.unlink()

num_train_episodes = len(train_episodes)
num_test_episodes = len(test_episodes)

for i in range(num_train_episodes + num_test_episodes):
    episode = (
        train_episodes[i]
        if i < num_train_episodes
        else test_episodes[i - num_train_episodes]
    )

    # first generated all the offsets and properties of the keep-out regions
    ko_traj_offsets = r.multivariate_normal(
        np.zeros(2), dist_ko_offsets, (num_ko_regions)
    )
    ko_vel_x = r.normal(0.0, dist_vel_x, (num_ko_regions))
    ko_vel_y = r.normal(0.0, dist_vel_y, (num_ko_regions))
    ko_accel_x = r.normal(0.0, dist_accel_x, (num_ko_regions))
    ko_accel_y = r.normal(0.0, dist_accel_y, (num_ko_regions))

    ko_radius = r.normal(dist_radius[0], dist_radius[1], (num_ko_regions))
    ko_vel_radius = r.normal(0.0, dist_vel_radius, (num_ko_regions))
    ko_accel_radius = r.normal(0.0, dist_accel_radius, (num_ko_regions))

    # now just need to find random points to place these keep-out regions along
    # the trajectory of the system (with appropriate offsets)

    # NOTE: this is highly inefficient but we just need to run it "once" during
    # dataset generation so it doesn't matter

    unique_rand_idxs = []
    while len(unique_rand_idxs) < num_ko_regions:
        idx = r.integers(0, len(episode["t_traj"]))
        if idx not in unique_rand_idxs:
            unique_rand_idxs.append(idx)

    ko_x = ko_traj_offsets[:, 0] + episode["x_traj"][unique_rand_idxs]
    ko_y = ko_traj_offsets[:, 1] + episode["y_traj"][unique_rand_idxs]

    # stores the results in the corresponding file

    file = (
        keep_out_train_dir.joinpath(train_files[i])
        if i < num_train_episodes
        else keep_out_test_dir.joinpath(test_files[i - num_train_episodes])
    )
    with h5py.File(file, "w") as f:
        f.create_dataset("ko_x", data=ko_x)
        f.create_dataset("ko_y", data=ko_y)
        f.create_dataset("ko_vel_x", data=ko_vel_x)
        f.create_dataset("ko_vel_y", data=ko_vel_y)
        f.create_dataset("ko_accel_x", data=ko_accel_x)
        f.create_dataset("ko_accel_y", data=ko_accel_y)
        f.create_dataset("ko_radius", data=ko_radius)
        f.create_dataset("ko_vel_radius", data=ko_vel_radius)
        f.create_dataset("ko_accel_radius", data=ko_accel_radius)